# Verify background clips are frog-free

Listen to each 3 s clip in `data/raw/background/` and mark whether any frog or toad is audible.
Answers are written to `labels/background_verification.csv` (tracked in git, so commit it when done).
Clips marked `yes`/`unsure` are excluded from the negative (background) class; you can resume any time.

In [1]:
from pathlib import Path

import pandas as pd
import soundfile as sf

from anuran.config import ROOT

BG = ROOT / "data" / "raw" / "background"
LABELS = ROOT / "labels"
LABELS.mkdir(exist_ok=True)

rows = []
for f in sorted(BG.glob("*.wav")):
    info = sf.info(f)
    rows.append({
        "clip_id": f.stem,
        "audio_path": str(f),
        "category": f.stem.rsplit("_", 1)[0],  # e.g. traffic_traffic_1521 -> traffic_traffic
        "start_time": 0.0,
        "end_time": round(info.duration, 3),
        "sample_rate": info.samplerate,
    })
clips = pd.DataFrame(rows)
clips_path = ROOT / "data" / "processed" / "review" / "background_clips.csv"
clips_path.parent.mkdir(parents=True, exist_ok=True)
clips.to_csv(clips_path, index=False)
print(len(clips), "clips")
print(clips["sample_rate"].value_counts().to_dict())
clips["category"].value_counts()

687 clips
{48000: 627, 32000: 54, 24000: 6}


category
airplane_airplane                  50
coyote_coyote                      50
device_static_device_static        50
engine_engine                      50
field_cricket_field_cricket        50
generator_generator                50
human_vocal_human_vocal            50
metal_clanging_metal_clanging      50
nutria_nutria                      50
power_tools_power_tools            50
traffic_traffic                    50
wind_wind                          50
thunder_thunder                    21
gun_gun                            19
dog_dog                            12
water_water                        11
fireworks_fireworks                 8
human_non_vocal_human_non_vocal     8
gun_fireworks                       5
branches_branches                   3
Name: count, dtype: int64

In [3]:
from jupyter_bioacoustic import BioacousticAnnotator

form = {
    "title": {"value": "Background clip: other species present?", "progress_tracker": True},
    "form": [
        {"checkbox": {"column": "other_species_present", "label": "Yes, other frog species audible in [[category]]"}},
        {"textbox": {"column": "notes", "label": "Notes (species if known)"}},
    ],
    "submission_buttons": {"line": True, "next": {"label": "Skip", "icon": True},
                           "submit": {"label": "Submit", "icon": True}},
}

ja = BioacousticAnnotator(
    data=str(clips_path),
    data_index_column="clip_id",
    data_start_time="start_time",
    data_end_time="end_time",
    audio_column="audio_path",
    display_columns=["category", "clip_id", "sample_rate"],
    form_config=form,
    output=str(LABELS / "background_verification.csv"),
    default_buffer=0,
    sort="category",
)
ja.open()

<IPython.core.display.Javascript object>

In [2085]:
# progress
out = LABELS / "background_verification.csv"
if out.exists():
    done = pd.read_csv(out)
    print(f"{done['clip_id'].nunique()} / {len(clips)} reviewed")
    print(done["frog_present"].value_counts().to_dict())

687 / 687 reviewed
{False: 650, True: 42}


In [ ]:
# Delete background clips marked as containing a frog (permanent).
v = pd.read_csv(LABELS / "background_verification.csv").drop_duplicates("clip_id", keep="last")
frog_ids = v.loc[v["frog_present"].astype(bool), "clip_id"]
to_delete = [BG / f"{cid}.wav" for cid in frog_ids if (BG / f"{cid}.wav").exists()]
print(f"{len(to_delete)} clips with frogs will be deleted from {BG}")

CONFIRM = False  # set True and re-run to delete
if CONFIRM:
    for f in to_delete:
        f.unlink()
    print(f"deleted {len(to_delete)}; {len(list(BG.glob('*.wav')))} clips remain")